<a href="https://colab.research.google.com/github/Misbah361/60-days-AI-challenge/blob/main/Day05Mini_Search_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

step 1: Installation

In [12]:
!pip install -q sentence-transformers numpy

step 2: Creating Dataset

In [13]:
import numpy as np, re
from sentence_transformers import SentenceTransformer

sentences = [
    "Rockets burn fuel to escape Earth's gravity and reach orbit.",
    "The International Space Station circles the planet every ninety minutes.",
    "Astronauts train underwater to simulate weightlessness.",
    "A black hole is a region where gravity is so strong that light cannot escape.",
    "Mars has the tallest volcano in the solar system, Olympus Mons.",
    "Telescopes collect faint light to reveal distant galaxies.",
    "The Moon's gravity causes ocean tides on Earth.",
    "Jupiter is a gas giant with a giant storm called the Great Red Spot.",
    "Satellites help us with weather forecasting and GPS navigation.",
    "A light-year is the distance light travels in one year.",
    "The Sun is a star that gives Earth heat and light.",
    "Comets are icy bodies that grow glowing tails near the Sun.",
    "Rovers drive across Mars searching for signs of ancient life.",
    "Neil Armstrong was the first human to walk on the Moon in 1969.",
    "Stars are born inside huge clouds of gas and dust called nebulae.",
    "Saturn's rings are made of ice and rock particles.",
    "Space debris from old satellites threatens active spacecraft.",
    "The Big Bang theory says the universe began from a hot, dense point.",
    "Reusable launch vehicles lower the cost of reaching space.",
    "Exoplanets are planets that orbit stars outside our solar system.",
]

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(sentences, normalize_embeddings=True)
print(len(sentences), embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

20 (20, 384)


step 3: Semantic search and Keyword search

In [14]:
def semantic_search(query, top_k=3):
    q = model.encode([query], normalize_embeddings=True)[0]
    scores = embeddings @ q
    idx = np.argsort(scores)[::-1][:top_k]
    return [(sentences[i], float(scores[i])) for i in idx]

def tokens(text):
    return set(re.findall(r"[a-z]+", text.lower()))

def keyword_search(query, top_k=3):
    q = tokens(query)
    scored = sorted(((s, len(q & tokens(s))) for s in sentences),
                    key=lambda x: x[1], reverse=True)[:top_k]
    return [(s, c) for s, c in scored if c > 0]

step 4: Comparison

In [15]:
queries = [
    "How do spacecraft leave the planet?",
    "Where did humans first step on another world?",
    "Dangers of junk floating around Earth",
]
for q in queries:
    print("=" * 70)
    print("QUERY:", q)
    print("\n-- Semantic top 3 --")
    for s, sc in semantic_search(q):
        print(f"  {sc:.3f}  {s}")
    print("\n-- Keyword top 3 --")
    res = keyword_search(q)
    if not res:
        print("  (no matches)")
    for s, c in res:
        print(f"  {c} shared words  {s}")

QUERY: How do spacecraft leave the planet?

-- Semantic top 3 --
  0.509  Rockets burn fuel to escape Earth's gravity and reach orbit.
  0.392  The International Space Station circles the planet every ninety minutes.
  0.365  Space debris from old satellites threatens active spacecraft.

-- Keyword top 3 --
  2 shared words  The International Space Station circles the planet every ninety minutes.
  1 shared words  Mars has the tallest volcano in the solar system, Olympus Mons.
  1 shared words  The Moon's gravity causes ocean tides on Earth.
QUERY: Where did humans first step on another world?

-- Semantic top 3 --
  0.386  Neil Armstrong was the first human to walk on the Moon in 1969.
  0.269  Rovers drive across Mars searching for signs of ancient life.
  0.222  Rockets burn fuel to escape Earth's gravity and reach orbit.

-- Keyword top 3 --
  2 shared words  Neil Armstrong was the first human to walk on the Moon in 1969.
  1 shared words  A black hole is a region where gravity is 

## Reflection: Semantic vs Keyword Search

**What I built:** A mini search engine over 20 space sentences. Each sentence is turned into a vector (a list of numbers) with `all-MiniLM-L6-v2`. A query is turned into a vector the same way, and the 3 sentences with the highest cosine similarity are returned. I also built a keyword search that only counts shared words, to compare.

**Query 1: "How do spacecraft leave the planet?"**
Semantic search ranked "Rockets burn fuel to escape Earth's gravity and reach orbit" first (score 0.509). That is the correct answer, even though it shares almost no words with the query. Keyword search ranked the Space Station sentence first, only because it shares "the planet". The rocket sentence did not appear in its top 3.

**Query 2: "Where did humans first step on another world?"**
Both searches put the Neil Armstrong sentence first. But keyword search got there through small common words like "first" and "on", and its 2nd and 3rd results (black hole, Moon tides) were unrelated. Semantic search's 2nd result (Mars rovers) was a more sensible match.

**Query 3: "Dangers of junk floating around Earth"**
Semantic search ranked the space debris sentence first (0.465), even though "junk" is not in it. Keyword search missed it completely. It returned rockets, tides, and the Sun, because they all share only the word "Earth".

**What surprised me:** Keyword search never said "no matches". It confidently returned results that looked related but were wrong, because it matches on common words and ignores meaning. Also, semantic scores were fairly low (0.2 to 0.5), but the ranking was still correct.

**Takeaway:** Keyword search finds the same words. Semantic search finds the same meaning. That is why RAG systems and AI assistants use embeddings to retrieve relevant information, since people rarely phrase a question the way the document does.